# Generative AI with LSTM - Text Generation

This notebook implements a character-level text generator using an LSTM (Long Short-Term Memory) neural network, trained on Shakespeare's complete works.

## Sections
1. Dataset Loading and Preprocessing
2. Model Design
3. Model Training
4. Text Generation
5. Architecture Experiments (Bonus)

## Setup - Import Libraries

In [ ]:
import os
import re
import requests
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 1. Dataset Loading and Preprocessing

**Dataset**: Shakespeare's Complete Works from [Project Gutenberg](https://www.gutenberg.org/ebooks/100)

**Steps**:
- Download the text file
- Convert to lowercase and remove punctuation
- Create character-level tokenization
- Prepare input-output sequence pairs

In [ ]:
# Download Shakespeare's works from Project Gutenberg
DATA_DIR = "data"
FILEPATH = os.path.join(DATA_DIR, "shakespeare.txt")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(FILEPATH):
    url = "https://www.gutenberg.org/cache/epub/100/pg100.txt"
    print(f"Downloading from {url}...")
    response = requests.get(url, timeout=60)
    with open(FILEPATH, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"Saved to {FILEPATH}")
else:
    print(f"Dataset already exists at {FILEPATH}")

# Load the raw text
with open(FILEPATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Raw text length: {len(raw_text):,} characters")

In [ ]:
# Preprocess the text

# Convert to lowercase
text = raw_text.lower()

# Remove Project Gutenberg header/footer
start_marker = "*** start of"
end_marker = "*** end of"

start_idx = text.find(start_marker)
if start_idx != -1:
    start_idx = text.find("\n", start_idx) + 1
else:
    start_idx = 0

end_idx = text.rfind(end_marker)
if end_idx == -1:
    end_idx = len(text)

text = text[start_idx:end_idx]

# Remove special characters, keep letters, spaces, and basic punctuation
text = re.sub(r"[^a-z\s.,;:!?\'\-]", "", text)
text = re.sub(r"\s+", " ", text).strip()

# Use first 200,000 characters for manageable training
MAX_CHARS = 200000
text = text[:MAX_CHARS]

print(f"Processed text length: {len(text):,} characters")
print(f"\nSample text (first 500 chars):\n{text[:500]}")

In [ ]:
# Create character-to-index and index-to-character mappings

chars = sorted(set(text))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Characters: {''.join(chars)}")

In [ ]:
# Prepare input-output sequence pairs
# Input: sequence of SEQ_LENGTH characters → Output: next character

SEQ_LENGTH = 40  # Number of characters in each input sequence

# Encode the text
encoded = [char_to_idx[ch] for ch in text]

# Create sequences
X, y = [], []
for i in range(len(encoded) - SEQ_LENGTH):
    X.append(encoded[i : i + SEQ_LENGTH])     # Input sequence
    y.append(encoded[i + SEQ_LENGTH])          # Next character (label)

X = np.array(X)
y = np.array(y)

print(f"Total sequences: {len(X):,}")
print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")

In [ ]:
# Split into training and validation sets (90% train, 10% validation)

split_idx = int(len(X) * 0.9)

X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print(f"Training samples:   {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")

---
## 2. Model Design

**Architecture**: Embedding → LSTM → Dropout → LSTM → Dropout → Dense (softmax)

- **Embedding layer**: Maps character indices to dense 256-dim vectors
- **2 LSTM layers**: 256 units each with dropout for regularization
- **Dense output**: Softmax over vocabulary for next-character prediction
- **Loss**: Sparse categorical crossentropy
- **Optimizer**: Adam (lr=0.001)

In [ ]:
# Build the LSTM model

EMBEDDING_DIM = 256
LSTM_UNITS = 256
DROPOUT_RATE = 0.2

model = Sequential(name="LSTM_TextGenerator")

# Embedding layer: maps character indices to dense vectors
model.add(Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM,
                    input_length=SEQ_LENGTH, name="embedding"))

# First LSTM layer (returns sequences for stacking)
model.add(LSTM(LSTM_UNITS, return_sequences=True, name="lstm_1"))
model.add(Dropout(DROPOUT_RATE, name="dropout_1"))

# Second LSTM layer
model.add(LSTM(LSTM_UNITS, return_sequences=False, name="lstm_2"))
model.add(Dropout(DROPOUT_RATE, name="dropout_2"))

# Dense output layer with softmax activation
model.add(Dense(vocab_size, activation="softmax", name="output"))

# Compile the model
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

model.summary()

---
## 3. Model Training

Training with:
- **Early Stopping**: Stops if validation loss doesn't improve for 5 epochs
- **Model Checkpoint**: Saves the best model based on validation loss

In [ ]:
# Setup training callbacks

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# Train the model
EPOCHS = 50
BATCH_SIZE = 128

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print(f"\nFinal Training Loss:   {history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.4f}")
print(f"Final Accuracy:        {history.history['accuracy'][-1]:.4f}")

---
## 4. Text Generation

Generate new text by feeding seed sequences to the trained model. We use **temperature sampling** to control creativity:
- **Low (0.2)**: Conservative, repetitive but coherent
- **Medium (0.5)**: Balanced
- **High (1.0)**: Creative but potentially less coherent

In [ ]:
def sample_with_temperature(predictions, temperature=0.5):
    """Sample character index from predictions with temperature scaling."""
    predictions = np.asarray(predictions).astype("float64")
    log_preds = np.log(predictions + 1e-8) / temperature
    exp_preds = np.exp(log_preds)
    probabilities = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(probabilities), p=probabilities)


def generate_text(model, seed_text, length=200, temperature=0.5):
    """Generate text from a seed sequence."""
    # Pad seed if shorter than sequence length
    if len(seed_text) < SEQ_LENGTH:
        seed_text = " " * (SEQ_LENGTH - len(seed_text)) + seed_text
    
    current_input = seed_text[-SEQ_LENGTH:]
    generated = list(seed_text)
    
    for _ in range(length):
        encoded = [char_to_idx.get(ch, 0) for ch in current_input]
        encoded = np.array(encoded).reshape(1, SEQ_LENGTH)
        
        predictions = model.predict(encoded, verbose=0)[0]
        next_idx = sample_with_temperature(predictions, temperature)
        next_char = idx_to_char.get(next_idx, " ")
        
        generated.append(next_char)
        current_input = current_input[1:] + next_char
    
    return "".join(generated)

In [ ]:
# Generate text with different seed inputs and temperatures

seeds = [
    text[:SEQ_LENGTH],                      # From dataset beginning
    "shall i compare thee to",              # Sonnet 18
    "to be or not to be",                   # Hamlet
    "friends, romans, countrymen",          # Julius Caesar
]

temperatures = [0.2, 0.5, 1.0]

for seed in seeds:
    print("=" * 60)
    print(f"SEED: \"{seed}\"")
    print("=" * 60)
    
    for temp in temperatures:
        output = generate_text(model, seed.lower(), length=200, temperature=temp)
        print(f"\n--- Temperature {temp} ---")
        print(output)
    print()

---
## 5. Bonus: Architecture Experiments

Compare different LSTM configurations to see how architecture affects text quality.

| Architecture | LSTM Layers | Units | Description |
|---|---|---|---|
| Shallow | 1 | 128 | Minimal model |
| Base | 2 | 256 | Default (trained above) |
| Deep | 3 | 256, 256, 128 | Deeper network |
| Wide | 2 | 512 | Wider layers |

In [ ]:
def build_experiment_model(lstm_configs, embedding_dim=256):
    """Build a model with custom LSTM configuration."""
    exp_model = Sequential()
    exp_model.add(Embedding(vocab_size, embedding_dim, input_length=SEQ_LENGTH))
    
    for i, units in enumerate(lstm_configs):
        return_seq = (i < len(lstm_configs) - 1)
        exp_model.add(LSTM(units, return_sequences=return_seq))
        exp_model.add(Dropout(0.2))
    
    exp_model.add(Dense(vocab_size, activation="softmax"))
    exp_model.compile(loss="sparse_categorical_crossentropy",
                      optimizer="adam", metrics=["accuracy"])
    return exp_model


# Define experiments
experiments = {
    "Shallow (1×128)": [128],
    "Base (2×256)":    [256, 256],
    "Deep (3 layers)":  [256, 256, 128],
    "Wide (2×512)":    [512, 512],
}

results = {}
seed_for_test = text[:SEQ_LENGTH]

for name, config in experiments.items():
    print(f"\n{'=' * 50}")
    print(f"Experiment: {name}")
    print(f"{'=' * 50}")
    
    exp_model = build_experiment_model(config)
    exp_model.summary()
    
    # Train for 10 epochs for comparison
    hist = exp_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=10, batch_size=128, verbose=1)
    
    # Generate sample
    sample = generate_text(exp_model, seed_for_test, length=200, temperature=0.5)
    
    results[name] = {
        "train_loss": hist.history["loss"][-1],
        "val_loss": hist.history["val_loss"][-1],
        "accuracy": hist.history["accuracy"][-1],
        "params": exp_model.count_params(),
        "sample": sample[:200]
    }
    
    del exp_model
    tf.keras.backend.clear_session()

In [ ]:
# Display experiment comparison

print("\n" + "=" * 80)
print("EXPERIMENT RESULTS SUMMARY")
print("=" * 80)
print(f"{'Architecture':<20} {'Params':>10} {'Train Loss':>12} {'Val Loss':>10} {'Accuracy':>10}")
print("-" * 65)

for name, res in results.items():
    print(f"{name:<20} {res['params']:>10,} {res['train_loss']:>12.4f} "
          f"{res['val_loss']:>10.4f} {res['accuracy']:>10.4f}")

print("\n" + "=" * 80)
print("GENERATED SAMPLES")
print("=" * 80)

for name, res in results.items():
    print(f"\n--- {name} ---")
    print(res["sample"])

---
## Observations & Conclusions

**Key findings from the experiments:**

1. **Shallow model (1×128)**: Fastest to train but produces less coherent text. The limited capacity means it struggles to capture long-range dependencies.

2. **Base model (2×256)**: Good balance of training time and output quality. The two stacked LSTM layers help capture more complex patterns in the text.

3. **Deep model (3 layers)**: Slightly better at capturing hierarchical patterns but takes longer to train. Diminishing returns beyond 2 layers for this dataset size.

4. **Wide model (2×512)**: Highest parameter count. Can memorize more patterns but also more prone to overfitting. Requires more data or regularization.

**Temperature effects:**
- `0.2`: Very conservative - produces repetitive but grammatically correct text
- `0.5`: Good balance of coherence and variety
- `1.0`: Most creative but occasionally produces nonsensical sequences

**Recommendation**: The base model (2×256) with temperature 0.5 provides the best trade-off for this dataset.